---
title: "Chapter 2: Crawling Methodology"
short_title: "2. Crawling Methodology"
---

This chapter describes how the crawler works and the tools and libraries used at each step. The complete source code is in the `src/` folder of the [group11-arxiv-project](https://github.com/mituleefam/group11-arxiv-project) repository, whose `README.md` explains how to set up the environment and reproduce the crawl.

## 1. Pipeline overview

`main.py` runs the following stages in order for one ID range:

```{mermaid}
flowchart LR
    R["Assigned ID range<br>start_month start_id end_month end_id"] --> D["discovery.py<br>arXiv API id_list"]
    D --> M["metadata.py<br>arXiv API, versioned IDs"]
    M --> DL["downloader.py + extractor.py<br>export.arxiv.org/src/&lt;id&gt;v&lt;N&gt;"]
    M --> RF["references.py<br>Semantic Scholar Graph API"]
    DL --> C[("corpus/&lt;yymm-id&gt;/")]
    RF --> C
    C --> V["validate.py<br>corpus_statistics.py"]
```

The source download and the reference crawl **run in parallel**, because they call two different servers (arXiv and Semantic Scholar), each with its own rate limit. Transient errors are not retried on the spot but collected: at the end of a pass, `main.py` runs further retry passes only for the stages that still have work. Final outcomes (a withdrawn version answering HTTP 404, a PDF-only version) are never retried.

## 2. Tools and libraries

| Component | Tools | Role |
| :--- | :--- | :--- |
| Discovery, metadata | arXiv API `https://export.arxiv.org/api/query` (Atom XML), `requests`, `xml.etree.ElementTree` | `id_list` lookups, reading the latest version suffix, per-version metadata |
| Source files | `https://export.arxiv.org/src/<id>v<N>`, `requests`, `tarfile`, `gzip`, `zipfile` | Per-version download, format detection, extraction of `.tex`/`.bib`/`.bbl` files |
| References | Semantic Scholar Graph API `/paper/batch`, `/paper/{id}/references` | Reference lists, `externalIds.ArXiv`, Semantic Scholar IDs |
| Reference submission dates | arXiv API, or the arXiv metadata snapshot on Kaggle (`ref_dates_kaggle.py`) | `ref_submission_date` (the `v1` submission date) of references with an arXiv ID |
| Routes to arXiv | Cloudflare WARP (`wgcf`, `wireproxy`, `requests[socks]`), Webshare proxies | Additional network paths to arXiv, see Section 7 |
| Execution on Colab | `chunked_crawl.py`, `snapshot.py`, Google Drive | Step-wise execution with a snapshot saved after each step ([Chapter 5](./05_performance.ipynb)) |
| Validation, measurement | `psutil`, `time.perf_counter` | RAM of the process tree, size of the data folder, wall time |
| Reporting | `pandas`, `plotly`, Jupyter Book | Tables and figures generated at book build time |

The `arxiv.py` library is not used: the crawler calls the endpoints directly with `requests`, so that the request pace, timeouts and retries of every stage are governed by one shared limiter (`rate_limiter.py`).

The statement presents several methods for discovery and source download. The choice follows the actual task, which is to resolve 25,000 given IDs and to download every version of each paper individually.

| Method | Used | Rationale |
| :--- | :--- | :--- |
| arXiv API, `id_list` lookup | Yes | Direct lookup by ID list; the returned identifier carries the latest version suffix, which gives the versions to download |
| OAI-PMH | No | Harvests by date range or category rather than by ID list, so a much larger set would have to be harvested and then filtered; resumption tokens also expire daily |
| arXiv metadata snapshot on Kaggle | Partly | Not used for discovery, since the snapshot may lack the newest papers. Used to look up the `v1` submission dates of cited papers without arXiv requests (`ref_dates_kaggle.py`), and to find the last ID of a month when a range spans several months (`assigned_ids.py`) |
| `export.arxiv.org/src/<id>v<N>` via `requests` | Yes | Per-version download from the endpoint required by the statement |
| `arxiv.py`, `arxiv-downloader` | No | `arxiv.py` 4.x no longer provides a download method; calling `requests` directly keeps every request under the same limiter |
| Amazon S3 (`s3://arxiv/src/`) | No | Requester-pays bucket (the downloader pays for the transfer), packaged by month with a total size of several TB, whereas only 25,000 papers are needed |

## 3. Entry discovery (`discovery.py`)

1. The list of arXiv IDs is generated from the four columns of the registration sheet (`start_month start_id end_month end_id`). A range may span several months; IDs issued since 2015 have five digits after the dot.
2. The arXiv API is called with `id_list` in batches of **100 IDs**, with `max_results` equal to the batch size (by default the API returns only 10 results).
3. The returned entries are not assumed to follow the input order: each entry is matched back to its ID through its own identifier.
4. The returned identifier carries the latest version suffix (for example `2604.05535v3`), from which the list `v1 … v3` to download is derived.
5. Each ID is written as one line of `work/paper_manifest.jsonl` with the status `success`, `not_found` or `api_error`. A failed batch is retried in a later pass and is never recorded as "not found".

## 4. Metadata (`metadata.py`)

The `metadata.json` file of each paper holds the fields of the `papers` table: `title`, `authors`, `submission_date` (the `v1` submission date), `revised_dates` (the dates of `v2, v3, …`) and `venue` (`journal_ref`, `null` when absent). Dates are normalized to ISO 8601. The metadata of **each version** is also stored in the `versions` list: the latest version comes from the discovery call, and older versions are looked up by versioned ID (`2604.05535v1`), 100 IDs per call.

## 5. Source download and extraction (`downloader.py`, `extractor.py`, `checkpoint.py`)

Each version goes through the following steps:

1. Download `https://export.arxiv.org/src/<id>v<N>` (never `arxiv.org`) with a descriptive User-Agent and a timeout, streaming into a temporary file `work/downloads/*.part`.
2. Check the HTTP status and compare the received size with `Content-Length`.
3. **Detect the actual format** from the first bytes of the file (magic number) instead of assuming `.tar.gz`:

| Detected format | Handling |
| :--- | :--- |
| `tar.gz`, `tar`, `zip` | Extract the `.tex`/`.bib`/`.bbl` members |
| `gzip-tex`, `plain-tex` | The whole paper is a single `.tex` file |
| `pdf` | No source: status `pdf_only`, empty folder |
| `docx`, `postscript` | No TeX source: status `no_tex`, empty folder |
| HTML page, including gzipped HTML | Treated as a server error page: status `failed`, retried in a later pass. In the actual data, the versions in this case turned out to be HTML submissions without TeX ([Chapter 6](./06_errors_limitations.ipynb), Section 3) |

4. Extract into a temporary folder `corpus/<yymm-id>/.<vN>.tmp/`, **writing only `.tex`, `.bib` and `.bbl` files** and keeping their relative paths and byte content unchanged. The `.bbl` files are kept following the instructor's Q&A ruling: for a paper with neither a `.bib` file nor `\bibitem` entries, the `.bbl` file is the only source of its references. An archive containing a `.bbl` file but no `.tex`/`.bib` file is not counted as a source (`no_tex`). Absolute paths, `..` components and links inside archives are rejected, so that an archive cannot write outside its version folder.
5. Verify the written files, rename the temporary folder to `v<N>/`, and only then record the status in `versions.json`. If the program stops midway, only `.part`/`.tmp` files remain, which the next run deletes and redoes. A completed version is never downloaded or overwritten again.

Sizes are measured on **the same basis, uncompressed bytes**: `original_bytes` is the total size of all files in the source archive and `kept_bytes` the total size of the kept `.tex`/`.bib`/`.bbl` files, so that `1 − kept_bytes / original_bytes` is the share of bytes removed. The size of the compressed download (`downloaded_bytes`) is recorded separately.

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils import (COLORS, FINAL_SOURCE, REF_OK, group, human_bytes, human_seconds,
                          load, members, meta, metric, mib, pct, provenance, show, table)
provenance()

Number of versions per format detected by `extractor.py`, over the whole corpus of the group:

In [ ]:
versions = load("versions")
fmt = (versions.assign(source_format=versions["source_format"].fillna("(undetermined)"))
       .groupby(["source_format", "status"]).size().reset_index(name="versions")
       .sort_values("versions", ascending=False))
fig = px.bar(fmt, x="source_format", y="versions", color="status", color_discrete_sequence=COLORS,
             title="Versions by detected source format and final status",
             labels={"source_format": "Detected format", "versions": "Versions", "status": "Status"})
show(fig)

## 6. References (`references.py`)

Semantic Scholar is queried with the prefix `arXiv:<id>` in three steps that minimize the number of requests:

1. `/paper/batch` with `paperId,referenceCount`, up to 500 papers per call, to learn which papers are indexed and how many references each one has.
2. `/paper/batch` with the reference fields (`title`, `authors`, `year`, `publicationDate`, `venue`, `externalIds`), to obtain the references of several papers in one call, grouped so that each call requests at most 8,000 references. This endpoint returns at most 9,999 references per call and **does not report an error when the limit is exceeded**, so a paper is kept only when its full `referenceCount` is returned.
3. The remaining papers are read page by page through `/paper/{id}/references`.

Each paper receives a status: `success`, `empty` (indexed by Semantic Scholar but listing no references), `unavailable` (references exist but are not shared), `not_found` (not indexed yet) or `api_error` (the API kept failing; nothing is written and the next run retries). **Genuinely empty reference lists** are thereby distinguished from **API errors**.

For each reference, the arXiv ID is read from `externalIds.ArXiv` (pre-2007 identifiers such as `hep-th/9901001` are kept as they are), together with the title, the authors and the Semantic Scholar ID. `ref_submission_date` is the arXiv `v1` submission date, looked up from the arXiv API or from the arXiv metadata snapshot on Kaggle; the Semantic Scholar `publicationDate` is **not** used in its place. Each reference also carries a `metadata_status`: `complete` (Semantic Scholar ID, title and authors present), `incomplete`, or `unresolved` (Semantic Scholar could not link the citation to a specific paper).

Output: `references_all.json` keeps **all** references, while `references.json` keeps only the references with an arXiv ID (the subset that becomes the `references` table in Milestone 3).

## 7. Rate limits and error handling (`rate_limiter.py`)

Every module sends its requests through one shared limiter, so the request pace holds even when several threads run in parallel:

| Server | Request pace | Error handling |
| :--- | :--- | :--- |
| arXiv API | At most 1 request every 3 s over a single connection (arXiv API Terms of Use); these calls also count toward the shared `export.arxiv.org` pace in the next row | As in the next row; a failed ID batch is retried in a later pass |
| `export.arxiv.org/src/` | Bursts of 4 requests 0.25 s apart followed by a 1 s pause (2 requests per second on average), following the arXiv *bulk data* page; at most 8 concurrent downloads | HTTP 429, or HTTP 503 and connection failures on several files: every arXiv request pauses for 5 s, doubled at each further refusal up to 2 minutes; after a 503, requests are spaced 6 s apart until 30 consecutive requests succeed. If arXiv refuses continuously for 30 minutes, the running step stops and all completed work stays saved. A file that keeps failing is deferred to a later pass without stopping the other files |
| Semantic Scholar | 1 request every 1.1 s with an API key, 1 request every 3.1 s without | HTTP 429: wait 2 s (or `Retry-After`), up to 12 times; 5xx and network errors: wait 5, 10, 20, 40 and 80 s |

The Semantic Scholar API key is read from the environment variable `S2_API_KEY` (Colab Secrets) and is never committed to the repository.

### Routes: network paths to arXiv

Requests to arXiv travel over several network paths (routes). The use of VPNs or IP rotation was permitted by the instructor in the project Q&A.

| Route | Setup | Requests carried |
| :--- | :--- | :--- |
| Colab | The runtime's own network | arXiv API calls and source downloads |
| WARP | Free Cloudflare WARP: `wgcf` registers an account and `wireproxy` runs WireGuard inside the runtime, exposing a SOCKS5 port | arXiv API calls and source downloads |
| Webshare | HTTP proxies loaded from Colab Secrets (`ARXIV_PROXIES`) | arXiv API calls only. Proxy bandwidth is limited (1 GB per month on the free plan), far below the volume of source files to download ([Chapter 4](./04_crawling_results.ipynb)) |
| AWS API Gateway | `requests-ip-rotator` | Supported by the code but **not enabled** in the benchmark run |

Routes **do not increase throughput**: all routes share the pace of the table above, since arXiv's limits apply to all of a user's machines as a whole. Their benefit is that when arXiv refuses a request (HTTP 429), **only that route pauses** for 15 s (doubled at each further refusal up to 5 minutes, and reset to 15 s once the route serves a request again), while the other routes continue at the normal pace. Every request pauses, as described in the table above, only when no route is available.

The table below is counted from the logs of each data folder (`download_log.csv`: one row per processing of one version; `arxiv_throttle_log.csv`: the pauses).

In [ ]:
routes = load("download_routes")
m = members()
dl = routes.pivot_table(index="member", columns="route", values="download_log_rows", aggfunc="sum", fill_value=0)
rp = routes.pivot_table(index="member", columns="route", values="route_pauses", aggfunc="sum", fill_value=0)
view = pd.DataFrame({"Member": m["member"]})
for r in [c for c in ("Colab", "WARP", "Webshare", "AWS") if c in dl.columns and dl[c].sum() > 0]:
    view[f"Downloads via {r}"] = view["Member"].map(dl[r]).fillna(0).astype(int)
for r in [c for c in ("Colab", "WARP", "Webshare", "AWS") if c in rp.columns]:
    view[f"{r} pauses"] = view["Member"].map(rp[r]).fillna(0).astype(int)
view["Global pauses"] = m["arxiv_pauses"].astype(int)
total = view.drop(columns="Member").sum()
view.loc[len(view)] = ["Group", *total.tolist()]
table(view, "Table 2.1. Source downloads by route and number of pauses, per member")
files = dl.sum()
print("Share of source downloads: " + ", ".join(f"{r} {pct(n / files.sum())}" for r, n in files.items() if n > 0) + ".")
print("The Webshare route carries arXiv API calls only, so it appears only in the pause columns.")

## 8. Validation and performance measurement

- `validate.py` checks the manifest, the metadata, every version (folder present, `kept_files` list matching the files on disk, no files other than `.tex`/`.bib`/`.bbl`, no empty files) and both reference files, and classifies each result as `PASS`, `WARN`, `PENDING` or `FAIL` ([Chapter 6](./06_errors_limitations.ipynb)).
- `corpus_statistics.py` computes the statistics from the final state on disk (`versions.json`, `references_all.json`) rather than summing log rows, since the logs also contain retries ([Chapter 4](./04_crawling_results.ipynb)).
- `benchmark.py` runs the crawler as a child process, sampling the RAM of the whole process tree every 2 s with `psutil` and the size of the data folder every 300 s ([Chapter 5](./05_performance.ipynb)).